In [ ]:
from dataclasses import dataclass
from pathlib import Path
import logging

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
    pipeline,
    set_seed,
 )
from peft import LoraConfig, TaskType, get_peft_model

# Logging in every script change: keep runtime visibility for each stage.
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("local_small_model")

@dataclass
class Config:
    # Local dataset and output paths for VS Code workflow.
    data_path: str = "data/raw/complaints-2026-07-04_02_07.csv"
    output_dir: str = "artifacts/qwen05b_lora_output"
    adapter_dir: str = "artifacts/qwen05b_lora_adapter"
    processed_data_dir: str = "data/processed"

    # Small model for local experimentation.
    model_name: str = "Qwen/Qwen2.5-0.5B"

    # Keep local run lightweight.
    train_rows: int = 400
    eval_rows: int = 80
    max_length: int = 256
    num_train_epochs: float = 1.0
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    logging_steps: int = 5
    eval_steps: int = 20
    save_steps: int = 40
    save_total_limit: int = 1
    seed: int = 42

    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05

cfg = Config()
set_seed(cfg.seed)

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.adapter_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.processed_data_dir).mkdir(parents=True, exist_ok=True)

logger.info("Config loaded for model: %s", cfg.model_name)

In [ ]:
df = pd.read_csv(cfg.data_path, dtype=str, low_memory=False)
logger.info("Loaded CSV rows=%s cols=%s", len(df), len(df.columns))

# Keep rows with useful narrative text.
text_col = "Consumer complaint narrative"
df = df[df[text_col].notna() & (df[text_col].str.len() > 80)].copy()
df = df[[text_col]].drop_duplicates().reset_index(drop=True)
logger.info("Rows after filtering: %s", len(df))

# Small local sample to stay within CPU/RAM limits.
train_df = df.iloc[: cfg.train_rows].copy()
eval_df = df.iloc[cfg.train_rows : cfg.train_rows + cfg.eval_rows].copy()

def build_prompt(text: str) -> str:
    return (
        "Instruction: Summarize the customer's finance complaint clearly and politely.\n"
        f"Input: {text}\n"
        "Response:"
    )

train_ds = Dataset.from_pandas(train_df.rename(columns={text_col: "text"}), preserve_index=False)
eval_ds = Dataset.from_pandas(eval_df.rename(columns={text_col: "text"}), preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    use_cache=False,
 )

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize_row(batch):
    prompts = [build_prompt(t) for t in batch["text"]]
    encoded = tokenizer(
        prompts,
        max_length=cfg.max_length,
        truncation=True,
        padding="max_length",
    )
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded

train_tok = train_ds.map(tokenize_row, batched=True, remove_columns=["text"])
eval_tok = eval_ds.map(tokenize_row, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    logging_steps=cfg.logging_steps,
    eval_steps=cfg.eval_steps,
    save_steps=cfg.save_steps,
    save_total_limit=cfg.save_total_limit,
    evaluation_strategy="steps",
    save_strategy="steps",
    report_to="none",
    fp16=False,
    bf16=False,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    load_best_model_at_end=False,
 )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=default_data_collator,
 )

logger.info("Starting local CPU training")
trainer.train()
logger.info("Saving LoRA adapter to %s", cfg.adapter_dir)
model.save_pretrained(cfg.adapter_dir)
tokenizer.save_pretrained(cfg.adapter_dir)

# Quick sanity inference
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,
    max_new_tokens=64,
    do_sample=False,
 )
test_prompt = build_prompt("I was charged twice for the same card payment and need help reversing one charge.")
result = generator(test_prompt)[0]["generated_text"]
print(result[:1200])